In [1]:
import pandas as pd
import requests
import time
import os
from dotenv import load_dotenv
from urllib.parse import quote

class NaverSearchAPI:
    def __init__(self):
        # .env 파일에서 환경변수 로드
        load_dotenv()
        
        self.client_id = os.getenv('Client_ID')
        self.client_secret = os.getenv('Client_Secret')
        
        # API 키 확인
        if not self.client_id or not self.client_secret:
            raise ValueError("API 키가 설정되지 않았습니다. .env 파일을 확인하세요.")
    
    def get_search_count(self, keyword, search_type="blog"):
        """네이버 검색 API로 검색 결과 수 가져오기"""
        
        # API URL 설정
        api_urls = {
            "blog": "https://openapi.naver.com/v1/search/blog.json",
            "news": "https://openapi.naver.com/v1/search/news.json",
            "cafe": "https://openapi.naver.com/v1/search/cafearticle.json",
            "web": "https://openapi.naver.com/v1/search/webkr.json"
        }
        
        url = api_urls.get(search_type, api_urls["blog"])
        
        # 헤더 설정
        headers = {
            'X-Naver-Client-Id': self.client_id,
            'X-Naver-Client-Secret': self.client_secret
        }
        
        # 파라미터 설정
        params = {
            'query': keyword,
            'display': 1,  # 1개만 가져와서 total만 확인
            'start': 1
        }
        
        try:
            response = requests.get(url, headers=headers, params=params)
            
            if response.status_code == 200:
                data = response.json()
                return data.get('total', 0)
            else:
                print(f"API 오류 ({search_type}): {response.status_code} - {keyword}")
                return 0
                
        except Exception as e:
            print(f"API 호출 오류: {e} - {keyword}")
            return 0
    
    def test_api_connection(self):
        """API 연결 테스트"""
        test_keyword = "한식 김치찌개"
        test_result = self.get_search_count(test_keyword)
        
        if test_result > 0:
            print(f"API 연결 테스트 성공: '{test_keyword}' 블로그 검색 결과 {test_result:,}개")
            return True
        else:
            print("API 연결 테스트 실패. .env 파일의 API 키를 확인하세요.")
            return False
    
    def collect_search_counts(self, keyword_data):
        """모든 키워드의 검색 결과 수 수집 (소분류 접두사 포함)"""
        
        results = []
        total_keywords = len(keyword_data)
        
        print(f"총 {total_keywords}개 키워드 검색 시작 (소분류 + 키워드 조합)")
        print("-" * 50)
        
        for i, data in enumerate(keyword_data, 1):
            subcategory = data['소분류']
            original_keyword = data['원본키워드']
            search_keyword = data['검색키워드']
            
            print(f"[{i}/{total_keywords}] {original_keyword} -> {search_keyword}")
            
            # 각 검색 타입별로 결과 수 가져오기
            blog_count = self.get_search_count(search_keyword, "blog")
            time.sleep(0.1)
            
            news_count = self.get_search_count(search_keyword, "news")
            time.sleep(0.1)
            
            cafe_count = self.get_search_count(search_keyword, "cafe")
            time.sleep(0.1)
            
            web_count = self.get_search_count(search_keyword, "web")
            time.sleep(0.1)
            
            # 결과 저장
            total_count = blog_count + news_count + cafe_count + web_count
            
            results.append({
                '소분류': subcategory,
                '원본키워드': original_keyword,
                '검색키워드': search_keyword,
                '블로그_검색수': blog_count,
                '뉴스_검색수': news_count,
                '카페_검색수': cafe_count,
                '웹_검색수': web_count,
                '총합': total_count
            })
            
            print(f"  블로그: {blog_count:,}, 뉴스: {news_count:,}, 카페: {cafe_count:,}, 웹: {web_count:,}")
            print(f"  총합: {total_count:,}")
            
            # API 호출 제한을 위한 대기
            time.sleep(0.5)
        
        return pd.DataFrame(results)

def load_keywords_from_csv():
    """CSV 파일에서 키워드와 소분류 읽기"""
    try:
        print("CSV 파일 읽기 중...")
        df = pd.read_csv('식당대12중53소132상세메뉴379분류.csv')
        
        keyword_data = []
        for _, row in df.iterrows():
            if pd.notna(row['상세메뉴']) and pd.notna(row['소분류']):
                subcategory = str(row['소분류']).strip()
                menus = str(row['상세메뉴']).strip()
                
                for menu in menus.split(','):
                    menu = menu.strip()
                    if menu:
                        keyword_data.append({
                            '소분류': subcategory,
                            '원본키워드': menu,
                            '검색키워드': f"{subcategory} {menu}"
                        })
        
        print(f"총 {len(keyword_data)}개 키워드 로딩 완료")
        return keyword_data
        
    except FileNotFoundError:
        print("오류: '식당대12중53소132상세메뉴379분류.csv' 파일을 찾을 수 없습니다.")
        print("현재 폴더에 CSV 파일이 있는지 확인하세요.")
        return None
    except Exception as e:
        print(f"CSV 파일 읽기 오류: {e}")
        return None

def process_all_keywords(keyword_data):
    """전체 키워드 처리"""
    print(f"\n전체 {len(keyword_data)}개 키워드를 처리합니다.")
    return keyword_data

def save_results_to_excel(results_df):
    """결과를 엑셀 파일로 저장"""
    # 결과 정렬 (총합 기준 내림차순)
    results_df = results_df.sort_values('총합', ascending=False)
    
    # 엑셀 파일로 저장
    output_filename = "소분류_키워드_검색량.xlsx"
    results_df.to_excel(output_filename, index=False)
    
    print("\n" + "=" * 50)
    print("수집 완료!")
    print(f"파일 저장: {output_filename}")
    print(f"총 처리된 키워드: {len(results_df)}개")
    
    # 상위 10개 키워드 출력
    print("\n상위 10개 키워드:")
    print("-" * 50)
    top_10 = results_df.head(10)
    for _, row in top_10.iterrows():
        print(f"{row['검색키워드']}: {row['총합']:,}개")
    
    # 통계 정보
    print(f"\n통계 정보:")
    print(f"평균 검색 결과 수: {results_df['총합'].mean():,.0f}개")
    print(f"최대 검색 결과 수: {results_df['총합'].max():,}개")
    print(f"최소 검색 결과 수: {results_df['총합'].min():,}개")
    
    return results_df

def create_env_file_template():
    """환경변수 파일 템플릿 생성"""
    env_template = """# 네이버 개발자센터에서 발급받은 API 키를 입력하세요
# https://developers.naver.com/apps/#/register

Client_ID=your_client_id_here
Client_Secret=your_client_secret_here
"""
    
    with open('.env', 'w', encoding='utf-8') as f:
        f.write(env_template)
    
    print(".env 파일 템플릿을 생성했습니다.")
    print("파일을 열어서 API 키를 입력한 후 다시 실행하세요.")

def main():
    print("네이버 검색 API를 사용한 소분류 + 키워드 검색량 조사")
    print("모든 키워드 앞에 소분류를 붙여서 검색합니다")
    print("=" * 60)
    
    # .env 파일 확인
    if not os.path.exists('.env'):
        print("오류: .env 파일이 없습니다.")
        print("현재 폴더에 .env 파일이 있는지 확인하세요.")
        return
    
    # API 클래스 초기화
    try:
        api = NaverSearchAPI()
    except ValueError as e:
        print(f"오류: {e}")
        print("\n.env 파일을 확인하세요:")
        print("Client_ID=your_client_id")
        print("Client_Secret=your_client_secret")
        return
    
    # API 연결 테스트
    print("\nAPI 연결 테스트 중...")
    if not api.test_api_connection():
        return
    
    # 키워드 로드
    keyword_data = load_keywords_from_csv()
    if keyword_data is None:
        return
    
    # 전체 키워드 처리
    selected_keyword_data = process_all_keywords(keyword_data)
    
    # 예상 소요 시간 계산
    estimated_time = len(selected_keyword_data) * 0.5 / 60  # 키워드당 0.5초 * 분 변환
    print(f"예상 소요 시간: 약 {estimated_time:.1f}분")
    
    # 처리 시작 확인
    start_confirm = input("\n처리를 시작하시겠습니까? (y/n): ").strip().lower()
    if start_confirm != 'y':
        print("처리를 중단합니다.")
        return
    
    # 검색 결과 수 수집
    try:
        print(f"\n전체 키워드 검색량 조사 시작...")
        results_df = api.collect_search_counts(selected_keyword_data)
        
        # 결과 저장 및 출력
        save_results_to_excel(results_df)
        
    except Exception as e:
        print(f"처리 중 오류 발생: {e}")
        return

if __name__ == "__main__":
    main()

네이버 검색 API를 사용한 소분류 + 키워드 검색량 조사
모든 키워드 앞에 소분류를 붙여서 검색합니다

API 연결 테스트 중...
API 연결 테스트 성공: '한식 김치찌개' 블로그 검색 결과 356,198개
CSV 파일 읽기 중...
총 381개 키워드 로딩 완료

전체 381개 키워드를 처리합니다.
예상 소요 시간: 약 3.2분

처리를 시작하시겠습니까? (y/n): y

전체 키워드 검색량 조사 시작...
총 381개 키워드 검색 시작 (소분류 + 키워드 조합)
--------------------------------------------------
[1/381] 제육볶음 -> 제육볶음 제육볶음
  블로그: 1,774,616, 뉴스: 30,151, 카페: 392,437, 웹: 2,609,163
  총합: 4,806,367
[2/381] 매운제육볶음 -> 제육볶음 매운제육볶음
  블로그: 202,459, 뉴스: 2,090, 카페: 26,301, 웹: 674,984
  총합: 905,834
[3/381] 두부제육볶음 -> 제육볶음 두부제육볶음
  블로그: 283,463, 뉴스: 2,433, 카페: 63,760, 웹: 884,054
  총합: 1,233,710
[4/381] 된장찌개 -> 찌개류 된장찌개
  블로그: 46,631, 뉴스: 1,081, 카페: 3,264, 웹: 39,263
  총합: 90,239
[5/381] 김치찌개 -> 찌개류 김치찌개
  블로그: 46,470, 뉴스: 1,491, 카페: 3,965, 웹: 64,083
  총합: 116,009
[6/381] 청국장찌개 -> 찌개류 청국장찌개
  블로그: 9,895, 뉴스: 166, 카페: 1,143, 웹: 26,011
  총합: 37,215
[7/381] 콩나물무침 -> 나물/반찬 콩나물무침
  블로그: 265,686, 뉴스: 1,179, 카페: 56,777, 웹: 658,100
  총합: 981,742
[8/381] 시금치나물 -> 나물/반찬 시금치나물
  블로그: 224,285, 뉴스

[89/381] 키조개 -> 조개구이 키조개
  블로그: 278,125, 뉴스: 3,549, 카페: 13,771, 웹: 304,542
  총합: 599,987
[90/381] 가리비 -> 조개구이 가리비
  블로그: 480,923, 뉴스: 2,071, 카페: 26,853, 웹: 476,039
  총합: 985,886
[91/381] 파전 -> 전류 파전
  블로그: 6,668, 뉴스: 192, 카페: 429, 웹: 187,085
  총합: 194,374
[92/381] 김치전 -> 전류 김치전
  블로그: 5,539, 뉴스: 167, 카페: 318, 웹: 518,045
  총합: 524,069
[93/381] 해물전 -> 전류 해물전
  블로그: 289, 뉴스: 21, 카페: 29, 웹: 303,780
  총합: 304,119
[94/381] 골뱅이무침 -> 무침류 골뱅이무침
  블로그: 1,517, 뉴스: 6, 카페: 351, 웹: 450,393
  총합: 452,267
[95/381] 오징어무침 -> 무침류 오징어무침
  블로그: 3,874, 뉴스: 59, 카페: 3,119, 웹: 1,518,261
  총합: 1,525,313
[96/381] 멍게무침 -> 무침류 멍게무침
  블로그: 553, 뉴스: 7, 카페: 71, 웹: 282,012
  총합: 282,643
[97/381] 곱창 -> 내장류 곱창
  블로그: 36,368, 뉴스: 233, 카페: 2,425, 웹: 360,349
  총합: 399,375
[98/381] 막창 -> 내장류 막창
  블로그: 23,438, 뉴스: 78, 카페: 1,212, 웹: 310,022
  총합: 334,750
[99/381] 순대 -> 내장류 순대
  블로그: 14,135, 뉴스: 66, 카페: 996, 웹: 389,091
  총합: 404,288
[100/381] 마른오징어 -> 마른안주 마른오징어
  블로그: 74,265, 뉴스: 1,947, 카페: 13,822, 웹: 575,366
  총합: 665,400
[1

[184/381] 쇼유라멘 -> 쇼유라멘 쇼유라멘
  블로그: 69,124, 뉴스: 443, 카페: 5,865, 웹: 72,144
  총합: 147,576
[185/381] 시오라멘 -> 쇼유라멘 시오라멘
  블로그: 19,879, 뉴스: 123, 카페: 1,330, 웹: 41,516
  총합: 62,848
[186/381] 맑은라멘 -> 쇼유라멘 맑은라멘
  블로그: 5,790, 뉴스: 39, 카페: 219, 웹: 31,213
  총합: 37,261
[187/381] 매운라멘 -> 매운라멘 매운라멘
  블로그: 417,193, 뉴스: 1,893, 카페: 19,475, 웹: 541,705
  총합: 980,266
[188/381] 마제소바 -> 매운라멘 마제소바
  블로그: 37,408, 뉴스: 31, 카페: 545, 웹: 191,611
  총합: 229,595
[189/381] 매운미소 -> 매운라멘 매운미소
  블로그: 56,594, 뉴스: 281, 카페: 2,769, 웹: 305,348
  총합: 364,992
[190/381] 츠케멘 -> 특수라멘 츠케멘
  블로그: 413, 뉴스: 4, 카페: 31, 웹: 28,159
  총합: 28,607
[191/381] 아부라소바 -> 특수라멘 아부라소바
  블로그: 109, 뉴스: 1, 카페: 9, 웹: 736
  총합: 855
[192/381] 탄탄멘 -> 특수라멘 탄탄멘
  블로그: 297, 뉴스: 11, 카페: 22, 웹: 2,555
  총합: 2,885
[193/381] 가츠동 -> 덮밥 가츠동
  블로그: 78,457, 뉴스: 567, 카페: 4,123, 웹: 230,558
  총합: 313,705
[194/381] 규동 -> 덮밥 규동
  블로그: 116,891, 뉴스: 1,289, 카페: 6,841, 웹: 91,438
  총합: 216,459
[195/381] 오야코동 -> 덮밥 오야코동
  블로그: 9,968, 뉴스: 130, 카페: 1,286, 웹: 12,412
  총합: 23,796
[196/

[277/381] 팟타이 -> 팟타이 팟타이
  블로그: 743,610, 뉴스: 6,450, 카페: 70,182, 웹: 293,879
  총합: 1,114,121
[278/381] 태국볶음면 -> 팟타이 태국볶음면
  블로그: 50,868, 뉴스: 604, 카페: 1,694, 웹: 46,849
  총합: 100,015
[279/381] 그린커리 -> 커리 그린커리
  블로그: 135,008, 뉴스: 11,021, 카페: 31,402, 웹: 549,627
  총합: 727,058
[280/381] 레드커리 -> 커리 레드커리
  블로그: 81,157, 뉴스: 2,708, 카페: 14,205, 웹: 452,971
  총합: 551,041
[281/381] 팬낭커리 -> 커리 팬낭커리
  블로그: 3, 뉴스: 0, 카페: 1, 웹: 0
  총합: 4
[282/381] 똠얌꿍 -> 똠얌 똠얌꿍
  블로그: 21,411, 뉴스: 85, 카페: 1,110, 웹: 8,244
  총합: 30,850
[283/381] 똠얌갈비 -> 똠얌 똠얌갈비
  블로그: 5,038, 뉴스: 43, 카페: 282, 웹: 4,826
  총합: 10,189
[284/381] 새콤매운국물 -> 똠얌 새콤매운국물
  블로그: 698, 뉴스: 7, 카페: 14, 웹: 586
  총합: 1,305
[285/381] 치킨커리 -> 커리 치킨커리
  블로그: 507,226, 뉴스: 7,757, 카페: 40,289, 웹: 1,184,139
  총합: 1,739,411
[286/381] 양고기커리 -> 커리 양고기커리
  블로그: 193,377, 뉴스: 1,580, 카페: 9,447, 웹: 487,367
  총합: 691,771
[287/381] 달커리 -> 커리 달커리
  블로그: 1,051, 뉴스: 50, 카페: 99, 웹: 689,261
  총합: 690,461
[288/381] 난 -> 난/로티 난
  블로그: 21,890, 뉴스: 352, 카페: 3,498, 웹: 288,397
  총합: 314,1

[369/381] 샐러드바 -> 샐러드바 샐러드바
  블로그: 3,042,671, 뉴스: 50,693, 카페: 204,733, 웹: 3,165,033
  총합: 6,463,130
[370/381] 샐러드뷔페 -> 샐러드바 샐러드뷔페
  블로그: 342,650, 뉴스: 9,430, 카페: 21,822, 웹: 797,299
  총합: 1,171,201
[371/381] 호텔뷔페 -> 호텔뷔페 호텔뷔페
  블로그: 1,399,934, 뉴스: 116,209, 카페: 467,001, 웹: 6,369,142
  총합: 8,352,286
[372/381] 브런치뷔페 -> 호텔뷔페 브런치뷔페
  블로그: 60,374, 뉴스: 5,743, 카페: 7,314, 웹: 697,967
  총합: 771,398
[373/381] 비건버거 -> 비건메인 비건버거
  블로그: 4,805, 뉴스: 322, 카페: 253, 웹: 279,380
  총합: 284,760
[374/381] 두부스테이크 -> 비건메인 두부스테이크
  블로그: 2,116, 뉴스: 101, 카페: 67, 웹: 275,038
  총합: 277,322
[375/381] 템페 -> 비건메인 템페
  블로그: 826, 뉴스: 14, 카페: 21, 웹: 5,799
  총합: 6,660
[376/381] 비건케이크 -> 비건디저트 비건케이크
  블로그: 82,275, 뉴스: 1,660, 카페: 3,264, 웹: 492,002
  총합: 579,201
[377/381] 두유아이스크림 -> 비건디저트 두유아이스크림
  블로그: 4,042, 뉴스: 125, 카페: 170, 웹: 253,274
  총합: 257,611
[378/381] 닭가슴살샐러드 -> 저칼로리 닭가슴살샐러드
  블로그: 44,285, 뉴스: 2,008, 카페: 5,627, 웹: 251,802
  총합: 303,722
[379/381] 퀴노아볼 -> 저칼로리 퀴노아볼
  블로그: 2,353, 뉴스: 107, 카페: 191, 웹: 185,343
  총합: 187,994